# Drive, github and requirements settings

In [ ]:
from google.colab import drive
from google.colab import userdata


drive.mount('/content/drive')
KEY = userdata.get('KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


set Github access token

In [ ]:
import os
from google.colab import userdata
github_access_token = userdata.get('GITHUB_TOKEN')

# Replace 'your_token_here' with your actual token and 'your_repo_url_here' with your repository URL
os.environ['GITHUB_TOKEN'] = github_access_token

repo_url = 'https://github.com/sustaz/principle_of_law_detection.git'
modified_url = repo_url.replace('https://', f'https://{os.environ["GITHUB_TOKEN"]}@')

Clone Repository

In [ ]:
!git clone {modified_url}

Cloning into 'principle_of_law_detection'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 47 (delta 22), reused 31 (delta 9), pack-reused 0
Receiving objects: 100% (47/47), 128.07 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (22/22), done.


Set github credentials

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/set_git_credentials.sh
!./principle_of_law_detection/bash_commands/set_git_credentials.sh

Syncronize notebook version

In [ ]:
!cp /content/drive/MyDrive/POLINE/gpt_notebook.ipynb /content/principle_of_law_detection/

In [ ]:
!cp /content/drive/MyDrive/POLINE/old_test_outputs.ipynb /content/principle_of_law_detection/

Add, commit, push code

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/add_commit_push.sh
!./principle_of_law_detection/bash_commands/add_commit_push.sh "cleaned gpt notebook and built library"

[main a736f27] cleaned gpt notebook and built library
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite gpt_notebook.ipynb (98%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 12.90 KiB | 4.30 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/sustaz/principle_of_law_detection.git
   c82380a..a736f27  main -> main


Install requirements

In [ ]:
!pip install -r '/content/principle_of_law_detection/requirements.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.2 MB/s eta 0:00:00


# Import libraries

In [ ]:
from principle_of_law_detection.src import gpt_utils as gu, utils as sr, text_preprocessing as tp, evaluation as ev
import json
import os
import pandas as pd

# Single experiments

In [ ]:
def prompt_single_trial1(txt):
  return f"""
  Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation or a previous paragraph.
        Not concern the application of facts to the current case.
        Not be what the referring court asks.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Classify each paragraph with Y if the paragraph is a JPOL, N if the paragraph is not a JPOL.
    Avoid any explanation into the output.
    Use the following format for the output:
      Paragraph number: class

      Here are a few examples of what is NOT a JPOL:
  37
  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.
  30
  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).
  54
  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

"""

system_prompt = "You are a judge with strong knowledge on tax law, expert about extracting Judicial Principles of Law (JPOLs) from legal judgments. You are very very skeptical and tend to say something is NOT a JPOL"

In [ ]:
def prompt_single_trial2(txt):
  return f"""
    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court;
  This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system.
  2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
  3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
  4. A JPOL is not a question concerning the application of facts to the current case AND
  5. A JPOL can be the endorsement of a precedent of a European Court AND
  6. A JPOL can be the endorsement of an Advocate General's statement AND
  7. A JPOL is not what the referring court asks.

  For each JPOL all the conditions must apply.

  Consider the following judgement, where each paragraph starts with a number:

  {txt}

  Check if each paragraph has the characteristics of a JPOL.

  Use the following format:
  Paragraph number: Y if JPOL.
  Paragraph number: N if not JPOL.
  Paragraph number: UND if it meets both criteria.

  Here are a few examples of what is NOT a JPOL:

  On the contrary, the consequence of such a condition is to exclude altogether any reduction of the taxable amount for VAT purposes in the case of unpaid claims arising during the six-month period preceding the declaration of insolvency of the debtor company concerned, even where those claims become definitively irrecoverable at the end of the insolvency proceedings. Such automatic refusal of the right to a reduction is contrary to the principle of the neutrality of VAT.

  As regards the context of which Article 132(1)(b) of the VAT Directive forms part, it is important to note that that provision must be read in the light of Article 134(a) of that directive, which requires, in any event, that the supply of goods or services concerned be essential to the transactions exempted within the scope of hospital and medical care (see, to that effect, judgments of 1 December 2005, Ygeia, C-394/04 and C-395/04, EU:C:2005:734, paragraph 26; of 14 June 2007, Horizon College, C-434/05, EU:C:2007:343, paragraph 38; and of 8 October 2020, Finanzamt D, C-657/19, EU:C:2020:811, paragraph 31).

  In the light of the discretion enjoyed by the Member States in that context, as noted in paragraph 40 above, the Court has held that the existence of the option provided for in the first paragraph of Article 133 of the VAT Directive supports the interpretation that it is for the national law of each Member State to lay down the rules according to which such recognition may be granted to establishments which request it, even if the fact that a Member State has not exercised that option does not affect the possibility that an establishment may be recognised for the purposes of granting the exemption referred to in Article 132(1)(b) of the VAT Directive (see, to that effect, judgment of 6 November 2003, Dornier, C-45/01, EU:C:2003:595, paragraphs 64 to 66).

  To recognise such an insurer as having that status would be tantamount to disregarding the principle of fiscal neutrality, since the VAT paid to the tax authorities would not be exactly proportional to the price actually received by the taxable customers who carried out the taxable transactions in question.
"""

system_prompt = ""

In [ ]:
def jpol_prompt_no_par_4(txt):
  prompt =  f""""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Only certain portions of the judgement text contain JPOLS, and these portions must be at least one complete sentence.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court; This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system
    2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
    3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
    4. A JPOL is not a question concerning the application of facts to the current case AND
    5. A JPOL can be the endorsement of a precedent of a European Court AND
    6. A JPOL can be the endorsement of an Advocate General's statement AND
    7. A JPOL is not what the referring court asks AND
    8. A JPOL is a complete sentence. It starts with a Capital letter and ends with a period (.)

    For each JPOL all the conditions must apply.

    #####

    Argumentative part of the judgment:

    {txt}

    #####

    For the output follow this format:
      - tag the portions of text which fit the JPOL definition between the tag <JPOL> </JPOL>.
      - Inside the tag, return only the first five and the last five words of the portion, separating them by dots.

    The output should be something like:

        <JPOL>Word1 word2 word3 word4 word5.....word-5 word-4 word-3 word-2 word-1.</JPOL>

    """

  return prompt


system_prompt_no_par = "You are an expert about Judicial Principles of Law (JPOLs) from legal judgments."

In [ ]:
response = gu.ask_gpt_2(prompt_single_trial_par(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset/Michael Winterhoff v Finanzamt Ulm and Jochen Eisenbeisvv Bundeszentralamt für Steuern.txt.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)

txt = tp.extract_text_between_markers(txt)

response = gu.ask_gpt_2(prompt_single_trial2(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

In [ ]:
print(response)

Paragraph 32: N
Paragraph 33: N
Paragraph 34: N
Paragraph 35: Y
Paragraph 36: Y
Paragraph 37: N
Paragraph 38: N
Paragraph 39: N
Paragraph 40: N
Paragraph 41: N
Paragraph 42: Y
Paragraph 43: Y
Paragraph 44: Y
Paragraph 45: Y
Paragraph 46: Y
Paragraph 47: Y
Paragraph 48: Y
Paragraph 49: Y
Paragraph 50: Y
Paragraph 51: Y
Paragraph 52: N
Paragraph 53: Y
Paragraph 54: Y
Paragraph 55: Y
Paragraph 56: N
Paragraph 57: N
Paragraph 58: N
Paragraph 59: N
Paragraph 60: N
Paragraph 61: N
Paragraph 62: N
Paragraph 63: N
Paragraph 64: N
Paragraph 65: Y
Paragraph 66: Y
Paragraph 67: Y
Paragraph 68: Y
Paragraph 69: Y
Paragraph 70: Y
Paragraph 71: Y


# MASSIVE EXPERIMENTS

In [ ]:
def jpol_prompt(txt):
  prompt =  f"""" Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation.
        Not concern the application of facts to the current case.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Use the following format for the output:
      Paragraph number: Y if JPOL.
      Paragraph number: N if not JPOL."""

  return prompt


system_prompt = "You are a professionist judge with strong knowledge on jurisdiction and tax law, expert about Judicial Principles of Law (JPOLs) from legal judgments. "

https://www.promptingguide.ai/techniques/prompt_chaining

In [ ]:
def jpol_prompt_few_shot(txt):
  prompt =  f""""

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation.
        Not concern the application of facts to the current case.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    ######

    To simplify your work, here are a few examples of what is NOT a JPOL, tagged with <OTHER>:

    <OTHER>
    By its first question, the referring court asks, in essence, whether the concept of SUBJECT_DEFINITION, within the meaning of ARTICLE of DIRECTIVE, must be interpreted as covering the activity provided by an entity, such as that at issue in the main proceedings, for the purpose of acquiring certain qualifications referred to in ARTICLE of DIRECTIVE.
    </OTHER>

    <OTHER>
    It is in the light of those considerations that the Court must examine whether the activity provided by an entity, such as that of the applicant in the main proceedings, for the purpose of acquiring certain qualifications referred to in ARTICLE of DIRECTIVE may be covered by the concept of SUBJECT_DEFINITION within the meaning of ARTICLE of DIRECTIVE.
    </OTHER>

    <OTHER>
    In the present case, the applicant in the main proceedings submits that the activity which it provides covers the transfer of both the practical and theoretical knowledge necessary for the purpose of acquiring certain qualifications and that the objective of such activity is not purely recreational, since possession of such qualifications is liable to meet, inter alia, professional needs. Therefore, the activity provided for that purpose is, it argues, covered by the concept of SUBJECT_DEFINITION referred to in ARTICLE of DIRECTIVE.
    </OTHER>

    <OTHER>
    It should be noted, however, that, even if it covers a range of practical and theoretical knowledge, the activity provided by an entity, such as that at issue in the main proceedings, nevertheless remains specialised tuition which does not amount, in itself, to the transfer of knowledge and skills covering a wide and diversified set of subjects or to their furthering and development which is characteristic of formal education.
    </OTHER>


    #####


    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    #####

    {txt}

    #####


    - Use the following format for the output:
    Paragraph number: JPOL
    Paragraph number: OTHER

    - Avoid any additional explanation in the output.
    """

  return prompt


system_prompt_few_shot = """You are an expert about Judicial Principles of Law (JPOLs) from legal judgments.  """

In [ ]:
def jpol_prompt_no_par(txt):
  prompt =  f""""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Of course, not all the text of the judgement contains JPOLS, but only some portions of it.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    - Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    - Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation.
        Not concern the application of facts to the current case.

    - Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    #####

    Argumentative part of the judgment:

    {txt}

    #####

    For the output follow this format:
      - tag the portions of text which fit the JPOL definition between the tag <JPOL> </JPOL>.
      - Inside the tag, return only the first five and the last five words of the portion, separating them by dots, for example, from this JPOL:

        According to the case-law of the Court, those exemptions constitute autonomous concepts of EU law which have the purpose of avoiding divergences in the application of the VAT
        system from one Member State to another.

        The output will be:

        <JPOL>According to the case-law.....one Member State to another</JPOL>

    """

  return prompt


system_prompt_no_par = "You are an expert about Judicial Principles of Law (JPOLs) from legal judgments."


def jpol_prompt_no_par_2(txt):
  prompt =  f""""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Of course, not all the text of the judgement contains JPOLS, but only some portions of it.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court; This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system.
    2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
    3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
    4. A JPOL is not a question concerning the application of facts to the current case AND
    5. A JPOL can be the endorsement of a precedent of a European Court AND
    6. A JPOL can be the endorsement of an Advocate General's statement AND
    7. A JPOL is not what the referring court asks.

    For each JPOL all the conditions must apply.

    #####

    Argumentative part of the judgment:

    {txt}

    #####

    For the output follow this format:
      - tag the portions of text which fit the JPOL definition between the tag <JPOL> </JPOL>.
      - Inside the tag, return only the first five and the last five words of the portion, separating them by dots, for example, from this JPOL:

        According to the case-law of the Court, those exemptions constitute autonomous concepts of EU law which have the purpose of avoiding divergences in the application of the VAT
        system from one Member State to another.

        The output will be:

        <JPOL>According to the case-law.....one Member State to another</JPOL>

    """

  return prompt


def jpol_prompt_no_par_4(txt):
  prompt =  f""""

    Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.
    Only certain portions of the judgement text contain JPOLS, and these portions must be at least one complete sentence.
    A JPOL (Judicial Principle of Law) should fit all the following conditions:

    1. A JPOL is a portion of text, extracted from the argumentative part of a judgement which contains the interpretation provided by the deciding court or provided in
     a previous decision and endorsed by the deciding court; This interpretation concerns a rule; or a general principle; or focuses on the consequences stemming from the application of a rule or a principle in a legal system
    2. A JPOL can be a citation of another JPOL taken from a previous judgement AND
    3. A JPOL is not a rephrase of the legislation or a previous paragraph AND
    4. A JPOL is not a question concerning the application of facts to the current case AND
    5. A JPOL can be the endorsement of a precedent of a European Court AND
    6. A JPOL can be the endorsement of an Advocate General's statement AND
    7. A JPOL is not what the referring court asks
    8. A JPOL is a complete sentence. It ends with a period (.)

    For each JPOL all the conditions must apply.

    #####

    Argumentative part of the judgment:

    {txt}

    #####

    For the output follow this format:
      - tag the portions of text which fit the JPOL definition between the tag <JPOL> </JPOL>.
      - Inside the tag, return only the first five and the last five words of the portion, separating them by dots, for example, from this JPOL:

    The output should be something like:

        <JPOL>word1 word2 word3 word4 word5.....word-5 word-4 word-3 word-2 word-1</JPOL>

    """

  return prompt


system_prompt_no_par = "You are an expert about Judicial Principles of Law (JPOLs) from legal judgments."

In [ ]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [ ]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Preprocessed_Judgements_Subset_noparagraph"
judgemnts = os.listdir(judgements_root)

In [ ]:
judgemnts

['A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml',
 'Autoridade Tributária e Aduaneira v Termas Sulfurosas de Alcafache SA.xml',
 'Michael Winterhoff v Finanzamt Ulm and Jochen Eisenbeisvv Bundeszentralamt für Steuern.xml',
 'Minister Finansów v Aviva Towarzystwo Ubezpieczeń na Życie S.A. w Warszawie.xml',
 'Boehringer Ingelheim RCV GmbH & Co. KG Magyarországi Fióktelepe v Nemzeti Adó- és Vámhivatal Fellebbviteli Igazgatósága.xml']

In [ ]:
prompt_name = "billaspries_4_no_par_generic_ex"
responses = []

for idx, judgement in enumerate(judgemnts):

  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  response = gu.ask_gpt_2(jpol_prompt_no_par_4(txt), system_prompt_no_par, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

  os.makedirs(f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/full_responses_{prompt_name}", exist_ok=True)
  sr.write_text_to_docx(response, f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/full_responses_{prompt_name}/{judgement}_full_response.docx")

  responses.append((response, judgement))

In [ ]:
def extract_paragraphs(text):
    # Regular expression to find pairs of number and answer (Y or N)
    pattern = r'(\d+): (JPOL|OTHER)'
    matches = re.findall(pattern, text)

    # Convert matches to list of tuples
    #pairs = [(int(num), answer) for num, answer in matches]

    return pd.DataFrame(matches, columns=['paragraph_number', 'label'])

In [ ]:
responses[1]

('<JPOL>It must be borne in.....their intended effect (judgment of</JPOL>\n\n<JPOL>The hospital and medical care.....C-700/17, EU:C:2019:753, paragraph</JPOL>\n\n<JPOL>In addition, medical services supplied.....judgment of 10 June 2010</JPOL>\n\n<JPOL>Since Article 132(1)(b) of.....the case-law cited)</JPOL>\n\n<JPOL>As regards the context of.....paragraph 31). As regards the</JPOL>\n\n<JPOL>In view of that objective.....Ygeia, C-394/04 and C-395/04</JPOL>\n\n<JPOL>For the purposes of determining.....the dispute before the</JPOL>\n\n<JPOL>Furthermore, it follows from the.....CopyGene, C-262/08, EU:C:2010:328</JPOL>\n\n<JPOL>Where that activity consists in.....the therapeutic objectives pursued</JPOL>\n\n<JPOL>By contrast, where all that.....the prescribed care may not</JPOL>\n\n<JPOL>If the referring court is.....the conditions laid down in</JPOL>\n\n<JPOL>In the light of the.....Article 132(1)(b) of the</JPOL>',
 'Autoridade Tributária e Aduaneira v Termas Sulfurosas de Alcafache SA.

Save results

In [ ]:
import re
dfs = []
for response, file_name in responses:
    dfs.append((extract_paragraphs(response.replace("<", "").replace(">", "")), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
#results_df.to_excel(f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}_remaining.xlsx", index=False)

NameError: name 'extract_paragraphs' is not defined

In [ ]:
results_df

,paragraph_number,label,file_name
0,16,OTHER,A & G
1,17,JPOL,A & G
2,18,JPOL,A & G
3,19,JPOL,A & G
4,20,JPOL,A & G
...,...,...,...
259,35,OTHER,Minis
260,36,OTHER,Minis
261,37,OTHER,Minis
262,38,JPOL,Minis


Read GT from Json

In [ ]:
ground_truths = []

for ann_file in annotations_files:

  ann_dict = json.load(open(os.path.join(jsons_root,ann_file)))


  for ann in ann_dict['annotations']:
    paragraph_number = ann['text'].split()[0]
    # split_par = str(int(paragraph_number) + 1)
    # txt_par = txt.split(split_par)
    label = ann['type']
    file_name = ann_file[:5]

    ground_truths.append((file_name, paragraph_number, label))

ground_truth_df = pd.DataFrame(ground_truths, columns=['file_name', 'paragraph_number', 'ground_truth'])

Read GT from excel

In [ ]:
ground_truth_df = pd.read_excel('/content/drive/MyDrive/POLINE/Annotazioni/CJUE_Taxable_Amount_Addendum_Piera.xlsx')
ground_truth_df['file_name'] = ground_truth_df['file_name'].apply(lambda x: x[:5])
ground_truth_df

# EVALUATION STRUCTURED IN PARAGRAPHS

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_metrics(ground_truth_df, results_df):

    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Mappare le etichette Y e N a JPOL e non-JPOL
    res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})

    # Unire i due DataFrame sui campi paragraph_number e file_name
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Creare le etichette binarie per il calcolo delle metriche
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Funzione per calcolare le metriche per ogni gruppo
    def calculate_metrics(group):
        precision = precision_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        recall = recall_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        f1 = f1_score(group['ground_truth_binary'], group['predicted_binary'], zero_division=0)
        return pd.Series({'precision': precision, 'recall': recall, 'f1': f1})

    # Calcolare le metriche per ogni valore di file_name
    metrics_by_filename = merged_df.groupby('file_name').apply(calculate_metrics).reset_index()

    return metrics_by_filename, merged_df

In [ ]:
def compute_total_metrics(ground_truth_df, results_df):
    res_df = results_df.copy()
    gt_df = ground_truth_df.copy()

    gt_df['paragraph_number'] = gt_df['paragraph_number'].astype(str)
    res_df['paragraph_number'] = res_df['paragraph_number'].astype(str)

    # Mappare le etichette Y e N a JPOL e non-JPOL
    res_df['label'] = res_df['label'].map({'JPOL': 'JPOL', 'OTHER': 'not_JPOL'})

    # Unire i due DataFrame sui campi paragraph_number e file_name
    merged_df = pd.merge(res_df, gt_df, on=['paragraph_number', 'file_name'], how='outer')

    # Creare le etichette binarie per il calcolo delle metriche
    merged_df['ground_truth_binary'] = merged_df['ground_truth'].apply(lambda x: 1 if x == 'JPOL' else 0)
    merged_df['predicted_binary'] = merged_df['label'].apply(lambda x: 1 if x == 'JPOL' else 0)

    # Calcolare precision, recall e f1-score
    precision = precision_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    recall = recall_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)
    f1 = f1_score(merged_df['ground_truth_binary'], merged_df['predicted_binary'], zero_division=0)

    return precision, recall, f1

In [ ]:
#results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/p_aspries_2_predictions.xlsx")

# Chiamare la funzione
metrics_by_filename, merged_df = compute_metrics(ground_truth_df, results_df)

# Mostrare i risultati
print(metrics_by_filename)

precision, recall, f1 = compute_total_metrics(ground_truth_df, results_df)

print(f'<--------------------->')

print(f'Total Precision: {precision:.2f}')
print(f'Total Recall: {recall:.2f}')
print(f'Total F1-Score: {f1:.2f}')

   file_name  precision    recall        f1
0      A & G   0.846154  1.000000  0.916667
1      Autor   0.727273  0.888889  0.800000
2      Boehr   0.619048  0.928571  0.742857
3      CS an   0.666667  0.307692  0.421053
4      DNB B   0.923077  0.750000  0.827586
5      ELVOS   0.421053  1.000000  0.592593
6      Euler   0.583333  1.000000  0.736842
7      Finan   0.615385  1.000000  0.761905
8      I Gmb   0.465116  0.952381  0.625000
9      Micha   0.625000  0.625000  0.625000
10     Minis   0.692308  0.642857  0.666667
<--------------------->
Total Precision: 0.61
Total Recall: 0.81
Total F1-Score: 0.70


In [ ]:
metrics_by_filename.to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/metrics_evaluation/p_aspries_2_predictions.xlsx")

- controllare qual è il valore di f1 dati solo gli esempi veri



- Provare predizioni di 5 in 5 paragrafi

- Creare un file univoco di predizioni, ground truth e testo per error analysis

In [ ]:
def extract_text_between_paragraphs(text, prev_par, next_par):
    # Define the patterns to search for
    pattern1 = re.compile(f"{prev_par}(.*?){next_par}", re.DOTALL)

    # Try to find matches for both patterns
    match1 = pattern1.search(text)

    # Return the matched text if found
    if match1:
        return match1.group(1).strip()
    else:
        return None

In [ ]:
par_txt = []
import re

for idx, row in merged_df.iterrows():

  text_file = [filename for filename in judgemnts if filename.startswith(row['file_name'])][0]

  with open(os.path.join(judgements_root, text_file)) as f:
    file = f.read()

  txt = "".join(file)

  txt = tp.extract_text_between_markers(txt).replace("\n", " ")
  txt = extract_text_between_paragraphs(txt, str(int(row['paragraph_number'])), str(int(row['paragraph_number']) + 1) )

  try:
    txt_par = txt.split(split_par)[0]
  except:
    print(str(int(row['paragraph_number'])))
    continue

  par_txt.append((row['file_name'], row['paragraph_number'], txt_par))

31
42
66
56
45
45
47
44
83
71


In [ ]:
par_txt_df = pd.DataFrame(par_txt, columns=['file_name', 'paragraph_number', 'text'])

In [ ]:
pd.merge(par_txt_df, merged_df, on=['file_name', 'paragraph_number']).to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/p_aspries_2_pred+gt.xlsx")

# EVALUATION UNSTRUCTURED

TODO BILLI:

*   Destruttura sentenze

TODO ASPRO:

*   Metriche AreaJPOL/AreaTesto
*   Map "5 words tags to paragraph" per comparare approccio strutturato e non






JPOL Ratio= Total number of words in the text / Total number of words tagged as JPOL​

\text{JPOL Ratio} = \frac{W_{\text{JPOL}}}{W_{\text{total}}}


Extract JPOL Sections: Use regex to find the text within JPOL tags.
Count JPOL Words: Split the JPOL text into parts around the dots. Count the words in the first and last parts.
Count Total Words: Remove JPOL tags from the total text and count the remaining words.
Add JPOL Word Count: Add the JPOL word count to the total word count.
Calculate Ratio: Compute the ratio of JPOL words to the total words.

In [ ]:
import re

def calculate_jpol_ratio(tagged_phrase, total_text, jpol_tag='<JPOL>', jpol_end_tag='</JPOL>'):
    # Extract JPOL sections
    jpol_texts = re.findall(f'{jpol_tag}(.*?){jpol_end_tag}', tagged_phrase, re.DOTALL)

    # Split JPOL text into beginning and end parts
    jpol_word_count = 0
    for jpol in jpol_texts:
        # Split into parts around the dots
        parts = jpol.split('...')
        if len(parts) == 2:
            # Count words in the first and last parts
            first_part_words = len(parts[0].split())
            last_part_words = len(parts[1].split())
            jpol_word_count += first_part_words + last_part_words

    # Remove JPOL tags from text to count total words
    clean_text = re.sub(f'{jpol_tag}.*?{jpol_end_tag}', '', total_text, flags=re.DOTALL)
    total_word_count = len(clean_text.split())

    # Add the JPOL word count to the total word count
    total_word_count += jpol_word_count

    # Calculate JPOL Ratio
    if total_word_count == 0:
        return 0

    jpol_ratio = jpol_word_count / total_word_count
    return round(jpol_ratio, 2)

In [ ]:
from docx import Document

f = open("/content/drive/MyDrive/POLINE/Results/preprocessed_input/full_responses_billaspries_4_no_par/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml_full_response.docx", 'rb')
document = Document(f)
f.close()

jpols_text = document.paragraphs[0].text

file = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset/A & G Fahrschul-Akademie GmbH v Finanzamt Wolfenbüttel.xml"

with open(file) as f:
    file = f.read()

judg_text = "".join(file)

judg_text = tp.extract_text_between_markers(judg_text)

In [ ]:
judg_text

'The first question\n\n16\n\nBy its first question, the referring court asks, in essence, whether the concept of ‘school or university education’, within the meaning of Article 132(1)(i) and (j) of Directive 2006/112, must be interpreted as covering motor vehicle driving tuition provided by a driving school, such as that at issue in the main proceedings, for the purpose of acquiring driving licences for vehicles in categories B and C1 referred to in Article 4(4) of Directive 2006/126.\n\n17\n\nArticle 132 of Directive 2006/112 provides for exemptions which, as indicated by the title of the chapter in which that provision features, are intended to encourage certain activities in the public interest. However, those exemptions do not cover every activity performed in the public interest, but only those listed in that provision and described in great detail (judgment of 4 May 2017, Brockenhurst College, C-699/15, EU:C:2017:344, paragraph 22 and the case-law cited).\n\n18\n\nAccording to th

In [ ]:
ratio = calculate_jpol_ratio(jpols_text, judg_text)
print(ratio)

0.13
